## 01 · Şirket içi belgelerin parçalanması
`belgeler/` klasöründeki 6 örnek (sentetik) iç belgeyi önce başlıklarına, sonra karakter sınırına göre parçalıyorum.

In [1]:
# Ortam hazırlığı: Colab'da repo klonlanır, yerelde notebooks/ klasöründen proje köküne geçilir
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("agentic-rag-turkish-docs").exists() and not Path("README.md").exists():
    !git clone -q https://github.com/alimdemir/agentic-rag-turkish-docs.git
if IN_COLAB and Path("agentic-rag-turkish-docs").exists():
    %cd agentic-rag-turkish-docs
    !pip -q install -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("çalışma klasörü:", Path.cwd().name)

çalışma klasörü: 03-agentic-rag-turkish-docs


In [2]:
import pandas as pd
from rag_utils import load_chunks

for size in (120, 350, 1000):
    ch = load_chunks(chunk_size=size, chunk_overlap=int(size * 0.15))
    print(f"chunk_size={size:<5} -> {len(ch):>3} parça, ortalama {sum(len(c.page_content) for c in ch)/len(ch):.0f} karakter")

chunk_size=120   ->  48 parça, ortalama 131 karakter
chunk_size=350   ->  20 parça, ortalama 243 karakter
chunk_size=1000  ->  20 parça, ortalama 243 karakter


In [3]:
chunks = load_chunks(chunk_size=350, chunk_overlap=50)
tablo = pd.DataFrame([{"kaynak": c.metadata["kaynak"], "bölüm": c.metadata.get("bolum"),
                       "karakter": len(c.page_content)} for c in chunks])
tablo.groupby("kaynak").agg(parça=("bölüm", "count"), ort_karakter=("karakter", "mean")).round(0)

,parça,ort_karakter
kaynak,,
aday_degerlendirme_sureci.md,4,240.0
gpu_sunucu_kullanimi.md,3,233.0
kvkk_mulakat_kayitlari.md,3,303.0
uzaktan_calisma_yonergesi.md,3,197.0
yeni_calisan_oryantasyonu.md,3,237.0
yillik_izin_politikasi.md,4,249.0


In [4]:
ornek = chunks[3]
print(ornek.metadata)
print("-" * 60)
print(ornek.page_content)

{'belge': 'Aday Değerlendirme Süreci', 'bolum': 'Yapay zekâ destekli analizler', 'kaynak': 'aday_degerlendirme_sureci.md', 'parca_no': 3}
------------------------------------------------------------
Aday Değerlendirme Süreci / Yapay zekâ destekli analizler
Mülakat panelindeki transkript ve duygu analizi çıktıları danışmana yardımcı bilgi olarak sunulur. Bu çıktılar tek başına işe alım kararı için kullanılmaz.


In [5]:
# Çok küçük parçada bilgi bağlamından kopuyor
kucuk = load_chunks(chunk_size=120, chunk_overlap=18)
print([c.page_content for c in kucuk if "90 gün" in c.page_content][0])

Mülakat Kayıtları ve Kişisel Verilerin Korunması / Saklama süreleri
Mülakat video ve ses kayıtları görüşme tarihinden itibaren en fazla 90 gün saklanır
